# 01 | Data understanding
Profile the supplied source as it exists. No cleaning or inferred corrections are applied in this notebook.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'restaurant-market-analysis']
PROJECT_ROOT = next(path for path in candidates if (path / 'data/raw/restaurant_data.csv').exists())
sys.path.insert(0, str(PROJECT_ROOT))
from src.data_loader import load_raw_data
raw = load_raw_data(PROJECT_ROOT / 'data/raw/restaurant_data.csv')
PROJECT_ROOT, raw.shape

In [ ]:
display(pd.DataFrame({'dtype': raw.dtypes.astype(str), 'missing': raw.isna().sum(), 'unique': raw.nunique(dropna=False)}))
print('Exact duplicate rows:', raw.duplicated().sum())
print('Duplicate restaurant IDs:', raw['Restaurant ID'].duplicated().sum())
print('Unique restaurant IDs:', raw['Restaurant ID'].nunique())

In [ ]:
for column in ['Country Code', 'City', 'Currency', 'Has Table booking', 'Has Online delivery', 'Is delivering now', 'Switch to order menu', 'Price range', 'Rating text']:
    print(f'\n{column}: {raw[column].value_counts(dropna=False).head(15).to_dict()}')
display(raw[['Longitude', 'Latitude', 'Average Cost for two', 'Aggregate rating', 'Votes']].describe(percentiles=[.01, .25, .5, .75, .95, .99]))
print('Zero coordinate pairs:', ((raw.Longitude == 0) & (raw.Latitude == 0)).sum())

## Initial data-quality notes
This source includes a UTF-8 BOM, missing cuisine labels, `(0, 0)` coordinate placeholders, zero ratings denoting `Not rated`, multiple currencies, and an invariant order-menu field. Cost comparisons must remain currency-specific; outliers are flags for review rather than grounds for deletion.